# ExtrusionExtraction: interactive Colab reconstruction (experimental)
Run the cells yourself in your own runtime. CPU only; no paid upgrade, tunnels, web service, background worker or automatic allocation. Free runtime availability is not guaranteed. References are uploaded into your personal temporary runtime. Export before disconnecting. Read the [Colab FAQ](https://research.google.com/colaboratory/faq.html) before using. Supply a reviewed source revision containing the headless task.

In [ ]:
from pathlib import Path
import os, subprocess, tempfile

# Set to the full reviewed commit containing the headless task.
REVISION = ''
assert len(REVISION) == 40 and all(c in '0123456789abcdef' for c in REVISION), 'Enter a reviewed Git commit SHA.'
root = Path(tempfile.mkdtemp(prefix='ee-'))
repo = root / 'repo'
subprocess.run(['git', 'init', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/HereLiesAz/ExtrusionExtraction.git'], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', '--depth=1', 'origin', REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == REVISION


## Build prerequisites
Install JDK 17, Android SDK 35 (required by the existing multi-target Gradle project), CMake 3.25+ and a C++23 compiler with `std::expected` for SkinTokens. Follow [Android SDK instructions](https://developer.android.com/tools/sdkmanager) and accept licenses yourself. Set JAVA_HOME and ANDROID_HOME below. No GPU accelerator is used.

In [ ]:
JAVA_HOME = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
ANDROID_HOME = os.environ.get('ANDROID_HOME', os.environ.get('ANDROID_SDK_ROOT', ''))
assert JAVA_HOME and (Path(JAVA_HOME) / 'bin/java').is_file(), 'Install JDK 17 or correct JAVA_HOME.'
assert ANDROID_HOME and (Path(ANDROID_HOME) / 'platforms/android-35/android.jar').is_file(), 'Install Android SDK 35 or correct ANDROID_HOME.'
os.environ.update(JAVA_HOME=JAVA_HOME, ANDROID_HOME=ANDROID_HOME)
os.environ['PATH'] = str(Path(JAVA_HOME)/'bin') + os.pathsep + os.environ['PATH']
subprocess.run(['java', '-version'], check=True)


In [ ]:
from google.colab import files
# Upload one .eejob exported by the app, or 4–24 individual reference images.
upload_dir = Path(tempfile.mkdtemp(prefix='upload-', dir=root))
previous = Path.cwd()
try:
    os.chdir(upload_dir)
    uploaded = files.upload()
finally:
    os.chdir(previous)
print(list(uploaded))


In [ ]:
import struct
ORDER = [] # only needed for individual images: exact filenames in viewing order
CELLS = 72
MASK = True
RIG = 'both' # none, template, learned, both; ignored for .eejob
refs = Path(tempfile.mkdtemp(prefix='refs-', dir=root))
output = root / ('output-' + refs.name)
command = ['./gradlew', ':desktopApp:reconstructHeadless', '--console=plain', '--no-daemon']
jobs = [name for name in uploaded if name.lower().endswith('.eejob')]
if jobs:
    assert len(jobs) == 1 and len(uploaded) == 1, 'Upload exactly one job file.'
    data = uploaded[jobs[0]]
    assert len(data) >= 16, 'Truncated job.'
    magic, cells, flags, count = struct.unpack_from('>4I', data)
    assert magic == 0x45454a31 and 32 <= cells <= 96 and 0 <= flags <= 7 and 4 <= count <= 24
    assert len(data) == 16 + count * 504 * 504 * 4, 'Invalid job length.'
    path = refs / 'reconstruction.eejob'
    path.write_bytes(data)
    args = ['--job', str(root/'models'), str(output), str(path)]
    needs_native = bool(flags & 4)
    print(f'{count} references, {cells} cells, mask={bool(flags & 1)}, template={bool(flags & 2)}, learned={needs_native}')
else:
    assert 4 <= len(ORDER) <= 24 and len(set(ORDER)) == len(ORDER) and set(ORDER) == set(uploaded)
    assert 32 <= CELLS <= 96 and RIG in ('none', 'template', 'learned', 'both')
    paths = []
    for i, name in enumerate(ORDER):
        path = refs / f'{i:02d}.image'
        path.write_bytes(uploaded[name])
        paths.append(str(path))
    args = [str(root/'models'), str(output), str(CELLS), 'mask' if MASK else 'no-mask', RIG, *paths]
    needs_native = RIG in ('learned', 'both')
if needs_native:
    native = subprocess.run(['python3', 'scripts/build_skintokens.py'], cwd=repo)
    if native.returncode == 0:
        command.append('-PskinTokensNativeDir=' + str(repo / 'build/skintokens/host-install'))
    else:
        print('Native build failed; proceeding with base and any template variant. Learned rig will report failure.')
command.append('--args=' + ' '.join(args))
status = subprocess.run(command, cwd=repo).returncode
print('Exit status:', status)
print('Successful exports:', [p.name for p in output.glob('*.glb')])
if status: print('At least one stage failed; inspect the log. Successful prior variants remain downloadable.')


In [ ]:
# Download completed files even if an optional rig failed.
exports = sorted(output.glob('*.glb'))
assert exports, 'No completed GLBs.'
for export in exports:
    files.download(str(export))
